# Última Ventana — entrenamiento, evaluación e inferencia

[Abrir este notebook en Google Colab](https://colab.research.google.com/github/v1ckybl/hackaton-equipo3/blob/feature/ultima-ventana/notebooks/02_entrenamiento_evaluacion_inferencia.ipynb)

Entrena desde cero con datos sintéticos generados en el mismo runtime, compara
dos baselines, exporta el modelo y prueba inferencia nominal, batch y temporal.


In [ ]:
# COLAB_CONFIG — editar solo esta celda
REPO_URL = "https://github.com/v1ckybl/hackaton-equipo3.git"
REPO_REF = "feature/ultima-ventana"
REPO_DIR = "/content/hackaton-equipo3"
OUTPUT_ROOT = "/content/ultima_ventana_outputs/02_entrenamiento_evaluacion"
ROWS = 10000
RANDOM_SEED = 42
N_ESTIMATORS = 200
MIN_ROC_AUC = 0.75
DOWNLOAD_OUTPUTS = False


## 1. Preparar el runtime


In [ ]:
# COLAB_SETUP — una sola preparación idempotente por runtime
import importlib
import os
import subprocess
import sys
from pathlib import Path

repo_dir = Path(REPO_DIR)
if repo_dir.exists() and not (repo_dir / ".git").is_dir():
    raise RuntimeError(f"{repo_dir} existe pero no es un clone Git válido")
if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(repo_dir)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", str(repo_dir)], check=True)
os.chdir(repo_dir)
for module_name in list(sys.modules):
    if module_name == "ultima_ventana_ml" or module_name.startswith("ultima_ventana_ml."):
        del sys.modules[module_name]
importlib.invalidate_caches()

commit = subprocess.run(
    ["git", "-C", str(repo_dir), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print(f"Entorno listo · Python {sys.version.split()[0]} · commit {commit}")


## 2. Ejecutar entrenamiento reproducible


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from ultima_ventana_ml import FEATURE_COLUMNS_V1, TARGET_NAME, RiskPredictor, run_pipeline

output_dir = Path(OUTPUT_ROOT)
dataset_path = output_dir / "training_dataset_synthetic_v1.csv"
model_dir = output_dir / "model"
summary = run_pipeline(
    ROWS, RANDOM_SEED, dataset_path, model_dir,
    n_estimators=N_ESTIMATORS, minimum_roc_auc=MIN_ROC_AUC,
)
display(pd.Series(summary, name="resultado"))


## 3. Evaluar modelo y baselines


In [ ]:
metrics = json.loads((model_dir / "metrics.json").read_text(encoding="utf-8"))
report = []
for key in ("baseline_dummy_test", "baseline_heuristic_test", "validation", "test"):
    report.append({"evaluación": key, "roc_auc": metrics[key]["roc_auc"], **metrics[key]["at_0_70"]})
display(pd.DataFrame(report).drop(columns="confusion_matrix").style.format(precision=3))

model = RiskPredictor.load(model_dir / "model.json", model_dir / "feature_schema.json")
data = pd.read_csv(dataset_path)
test = data.loc[data["split"] == "TEST"].copy()
scored = model.predict_many(test)


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
y_true, y_score = test[TARGET_NAME], scored["risk_score"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
RocCurveDisplay.from_predictions(y_true, y_score, ax=axes[0], name="XGBoost")
ConfusionMatrixDisplay.from_predictions(y_true, y_score >= 0.70, ax=axes[1], colorbar=False)
axes[1].set_title("Matriz de confusión · 0.70")
plt.tight_layout(); plt.show()

importance = pd.Series(model.feature_importances, index=FEATURE_COLUMNS_V1).sort_values()
importance.plot.barh(title="Importancia de features"); plt.tight_layout(); plt.show()


## 4. Probar predictor individual y batch


In [ ]:
one = {
    "rain_24h_mm": 61.4, "rain_72h_mm": 138.2,
    "forecast_rain_6h_mm": 34.0, "forecast_rain_12h_mm": 62.0,
    "elevation_mean_m": 48.7, "slope_mean_pct": 0.42,
    "water_coverage_100m_ratio": 0.27,
}
print(f"Predicción individual: {model.predict(one):.3f}")
batch = model.predict_many(test.head(8))
batch.to_csv(output_dir / "batch_predictions.csv", index=False)
display(batch[["synthetic_sample_id", "risk_score", "risk_level"]])


## 5. Probar serie temporal y hora crítica


In [ ]:
from ultima_ventana_ml import (
    calculate_last_safe_departure, find_critical_time, generate_demo_timeline,
)
timeline = model.predict_many(generate_demo_timeline())
critical_time = find_critical_time(timeline)
if critical_time is None:
    raise RuntimeError("El escenario controlado no alcanzó el umbral crítico")
last_departure = calculate_last_safe_departure(critical_time, 80, 40)
timeline.to_csv(output_dir / "timeline_predictions.csv", index=False)
display(timeline[["prediction_time", "risk_score", "risk_level"]])
print(f"Hora crítica: {critical_time} · Última salida demostrativa: {last_departure}")


## 6. Empaquetar artefactos


In [ ]:
import shutil
archive_path = shutil.make_archive(str(output_dir), "zip", root_dir=output_dir)
print(f"Artefactos empaquetados: {archive_path}")
if DOWNLOAD_OUTPUTS:
    from google.colab import files
    files.download(archive_path)


El score es experimental y está entrenado contra labels sintéticos; no expresa precisión real.
